# Week 16: Deep Learning & LLMs — Applied Statistics with Python (2026)

In our final content week, we venture into **deep learning** — the technology behind image recognition, language translation, and AI assistants. We'll build neural networks from scratch, use PyTorch to train real models, and explore how **Large Language Models (LLMs)** work.

| # | Topic |
|---|-------|
| 1 | From ML to Deep Learning |
| 2 | The Perceptron & Neurons |
| 3 | Neural Networks from Scratch |
| 4 | Training: Backpropagation & Gradient Descent |
| 5 | Building Neural Networks with PyTorch |
| 6 | Convolutional Neural Networks (CNNs) — Overview |
| 7 | Recurrent Neural Networks (RNNs) — Overview |
| 8 | The Transformer Architecture |
| 9 | Large Language Models (LLMs) |
| 10 | Using LLMs via APIs |
| 11 | Practical Example: Image Classification with PyTorch |
| 12 | Summary & Homework |

## Imports

We import PyTorch for deep learning, along with our standard data science libraries.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris, load_digits, make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# PyTorch imports
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader, TensorDataset
    TORCH_AVAILABLE = True
    print(f'PyTorch version: {torch.__version__}')
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Device: {device}')
except ImportError:
    TORCH_AVAILABLE = False
    print('PyTorch not installed. Install with: pip install torch')
    print('We will use NumPy implementations instead.')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')
np.random.seed(42)

print('All imports successful!')

---
## 1. From ML to Deep Learning

Deep learning is a subset of machine learning that uses **neural networks with many layers** to learn hierarchical representations of data.

### Why Deep Learning?

| Traditional ML | Deep Learning |
|---------------|---------------|
| Manual feature engineering | Learns features automatically |
| Works well on structured data | Excels on images, text, audio |
| Smaller datasets (100s–10Ks) | Larger datasets (10Ks–millions) |
| Interpretable (often) | Black box (often) |
| Fast training | Needs GPUs for large models |

### Deep Learning Milestones

| Year | Breakthrough |
|------|-------------|
| 2012 | AlexNet wins ImageNet (CNNs go mainstream) |
| 2014 | GANs for image generation |
| 2017 | Transformer architecture ("Attention Is All You Need") |
| 2018 | BERT (bidirectional language understanding) |
| 2020 | GPT-3 (175B parameter language model) |
| 2022–25 | ChatGPT, GPT-4, Claude — LLMs go mainstream |

---
## 2. The Perceptron & Neurons

The **perceptron** is the simplest neural network — a single artificial neuron. It computes a weighted sum of inputs, adds a bias, and passes the result through an activation function.

$$y = f\left(\sum_{i=1}^{n} w_i x_i + b\right) = f(\mathbf{w}^T \mathbf{x} + b)$$

### Common Activation Functions

| Function | Formula | Range | Use |
|----------|---------|----------|-----|
| **Sigmoid** | $\frac{1}{1+e^{-x}}$ | (0, 1) | Binary output, gates |
| **Tanh** | $\frac{e^x - e^{-x}}{e^x + e^{-x}}$ | (-1, 1) | Hidden layers (legacy) |
| **ReLU** | $\max(0, x)$ | [0, ∞) | Default for hidden layers |
| **Softmax** | $\frac{e^{x_i}}{\sum_j e^{x_j}}$ | (0, 1), sum=1 | Multi-class output |

### 2.1 Visualizing Activation Functions

Let's plot and compare the most important activation functions.

In [ ]:
x = np.linspace(-5, 5, 200)

# Define activation functions
sigmoid = 1 / (1 + np.exp(-x))
tanh = np.tanh(x)
relu = np.maximum(0, x)
leaky_relu = np.where(x > 0, x, 0.01 * x)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

activations = [
    ('Sigmoid', sigmoid, 'blue'),
    ('Tanh', tanh, 'green'),
    ('ReLU', relu, 'red'),
    ('Leaky ReLU', leaky_relu, 'purple'),
]

for idx, (name, y, color) in enumerate(activations):
    axes[idx].plot(x, y, color=color, lw=2)
    axes[idx].axhline(0, color='gray', lw=0.5)
    axes[idx].axvline(0, color='gray', lw=0.5)
    axes[idx].set_title(name, fontsize=13)
    axes[idx].set_xlabel('x')
    axes[idx].set_ylabel('f(x)')
    axes[idx].grid(True, alpha=0.3)

plt.suptitle('Neural Network Activation Functions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('ReLU is the default choice for hidden layers:')
print('  - Computationally efficient (just max(0,x))')
print('  - Avoids vanishing gradient problem')
print('  - Sparse activation (many zeros)')

### 2.2 A Single Perceptron (AND / OR / XOR)

Let's build a perceptron from scratch and see what it can and cannot learn.

In [ ]:
class Perceptron:
    """Simple perceptron with sigmoid activation."""
    
    def __init__(self, n_inputs, lr=0.5):
        self.weights = np.random.randn(n_inputs) * 0.1
        self.bias = 0.0
        self.lr = lr
    
    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))
    
    def predict(self, X):
        z = X @ self.weights + self.bias
        return self.sigmoid(z)
    
    def train(self, X, y, epochs=1000):
        losses = []
        for _ in range(epochs):
            # Forward pass
            y_hat = self.predict(X)
            error = y - y_hat
            
            # Gradient descent update
            self.weights += self.lr * (X.T @ (error * y_hat * (1 - y_hat))) / len(y)
            self.bias += self.lr * np.mean(error * y_hat * (1 - y_hat))
            
            loss = -np.mean(y * np.log(y_hat + 1e-8) + (1-y) * np.log(1-y_hat + 1e-8))
            losses.append(loss)
        return losses

# Logic gates
X_logic = np.array([[0,0], [0,1], [1,0], [1,1]])

gates = {
    'AND': np.array([0, 0, 0, 1]),
    'OR':  np.array([0, 1, 1, 1]),
    'XOR': np.array([0, 1, 1, 0]),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (name, y_target) in enumerate(gates.items()):
    p = Perceptron(2, lr=5.0)
    losses = p.train(X_logic, y_target, epochs=5000)
    preds = (p.predict(X_logic) > 0.5).astype(int)
    acc = np.mean(preds == y_target)
    
    # Decision boundary
    xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 100), np.linspace(-0.5, 1.5, 100))
    Z = p.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    axes[idx].contourf(xx, yy, Z, levels=20, cmap='RdYlBu', alpha=0.4)
    axes[idx].scatter(X_logic[y_target==0, 0], X_logic[y_target==0, 1],
                       c='blue', s=100, marker='o', edgecolors='black', label='0')
    axes[idx].scatter(X_logic[y_target==1, 0], X_logic[y_target==1, 1],
                       c='red', s=100, marker='s', edgecolors='black', label='1')
    axes[idx].set_title(f'{name} Gate (Acc={acc:.0%})', fontsize=12)
    axes[idx].legend()

plt.suptitle('Single Perceptron: What It Can Learn', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('A single perceptron can learn AND and OR (linearly separable).')
print('It CANNOT learn XOR — we need multiple layers for that!')

---
## 3. Neural Networks from Scratch

A **multi-layer neural network** (MLP) stacks multiple layers of neurons. With hidden layers, it can learn non-linear patterns like XOR.

### Architecture:
- **Input layer**: receives features (no computation)
- **Hidden layer(s)**: learn intermediate representations
- **Output layer**: produces final prediction

### 3.1 Two-Layer Network for XOR

Let's build a 2-layer network from scratch to solve XOR.

In [ ]:
class SimpleNeuralNetwork:
    """
    2-layer neural network built from scratch with NumPy.
    Architecture: Input → Hidden (ReLU) → Output (Sigmoid)
    """
    
    def __init__(self, n_input, n_hidden, n_output, lr=0.1):
        # Initialize weights with small random values
        self.W1 = np.random.randn(n_input, n_hidden) * 0.5
        self.b1 = np.zeros(n_hidden)
        self.W2 = np.random.randn(n_hidden, n_output) * 0.5
        self.b2 = np.zeros(n_output)
        self.lr = lr
    
    def relu(self, z):
        return np.maximum(0, z)
    
    def relu_deriv(self, z):
        return (z > 0).astype(float)
    
    def sigmoid(self, z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
    
    def forward(self, X):
        """Forward pass through the network."""
        self.z1 = X @ self.W1 + self.b1       # hidden pre-activation
        self.a1 = self.relu(self.z1)            # hidden activation
        self.z2 = self.a1 @ self.W2 + self.b2  # output pre-activation
        self.a2 = self.sigmoid(self.z2)         # output activation
        return self.a2
    
    def backward(self, X, y):
        """Backpropagation: compute gradients and update weights."""
        m = len(y)
        y = y.reshape(-1, 1)
        
        # Output layer gradient
        dz2 = self.a2 - y                      # (prediction - target)
        dW2 = self.a1.T @ dz2 / m
        db2 = np.mean(dz2, axis=0)
        
        # Hidden layer gradient (chain rule!)
        dz1 = (dz2 @ self.W2.T) * self.relu_deriv(self.z1)
        dW1 = X.T @ dz1 / m
        db1 = np.mean(dz1, axis=0)
        
        # Update weights
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1
    
    def train(self, X, y, epochs=5000):
        """Train the network."""
        losses = []
        for epoch in range(epochs):
            y_hat = self.forward(X)
            self.backward(X, y)
            loss = -np.mean(y*np.log(y_hat+1e-8) + (1-y)*np.log(1-y_hat+1e-8))
            losses.append(loss)
        return losses

# Solve XOR with 2-layer network!
np.random.seed(42)
nn_xor = SimpleNeuralNetwork(n_input=2, n_hidden=4, n_output=1, lr=1.0)
losses = nn_xor.train(X_logic, gates['XOR'], epochs=10000)

preds = nn_xor.forward(X_logic)
print('XOR with 2-layer neural network:')
for i in range(4):
    print(f'  Input: {X_logic[i]} → Output: {preds[i,0]:.4f} → Predicted: {int(preds[i,0]>0.5)} (True: {gates["XOR"][i]})')

# Plot loss curve and decision boundary
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(losses, 'b-', lw=1)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')

xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 200), np.linspace(-0.5, 1.5, 200))
Z = nn_xor.forward(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
axes[1].contourf(xx, yy, Z, levels=20, cmap='RdYlBu', alpha=0.5)
axes[1].scatter(X_logic[gates['XOR']==0, 0], X_logic[gates['XOR']==0, 1],
                 c='blue', s=100, marker='o', edgecolors='black', label='0')
axes[1].scatter(X_logic[gates['XOR']==1, 0], X_logic[gates['XOR']==1, 1],
                 c='red', s=100, marker='s', edgecolors='black', label='1')
axes[1].set_title('XOR Decision Boundary (2-layer NN)')
axes[1].legend()

plt.tight_layout()
plt.show()

print('\nWith hidden layers, the network learns the non-linear XOR function!')

---
## 4. Training: Backpropagation & Gradient Descent

**Backpropagation** is the algorithm used to compute gradients in neural networks. Combined with **gradient descent**, it adjusts weights to minimize the loss.

### The Training Loop

1. **Forward pass**: compute predictions
2. **Compute loss**: measure prediction error
3. **Backward pass**: compute gradients (∂Loss/∂weights)
4. **Update weights**: $w \leftarrow w - \eta \cdot \nabla L$
5. Repeat

### Gradient Descent Variants

| Variant | Batch Size | Speed | Stability |
|---------|-----------|-------|----------|
| **Batch GD** | All data | Slow per step | Stable |
| **Stochastic GD (SGD)** | 1 sample | Fast per step | Noisy |
| **Mini-batch GD** | 32–256 samples | Balanced | Balanced |
| **Adam** | Mini-batch + adaptive LR | Fast | Very stable |

### 4.1 Visualizing Gradient Descent

Let's visualize how different optimizers navigate a loss landscape.

In [ ]:
# Visualize gradient descent on a 2D loss surface
def rosenbrock(x, y):
    """Rosenbrock function — a classic optimization test function."""
    return (1 - x)**2 + 100 * (y - x**2)**2

def grad_rosenbrock(x, y):
    """Gradient of Rosenbrock function."""
    dx = -2 * (1 - x) + 200 * (y - x**2) * (-2 * x)
    dy = 200 * (y - x**2)
    return dx, dy

def gradient_descent_path(start, lr, n_steps):
    """Run gradient descent and record the path."""
    path = [start]
    x, y = start
    for _ in range(n_steps):
        dx, dy = grad_rosenbrock(x, y)
        x -= lr * dx
        y -= lr * dy
        path.append((x, y))
    return np.array(path)

# Create loss surface
xx = np.linspace(-2, 2, 200)
yy = np.linspace(-1, 3, 200)
XX, YY = np.meshgrid(xx, yy)
ZZ = rosenbrock(XX, YY)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
start = (-1.5, 2.0)

lrs = [0.0001, 0.001, 0.005]
titles = ['LR=0.0001 (too slow)', 'LR=0.001 (good)', 'LR=0.005 (unstable)']

for idx, (lr, title) in enumerate(zip(lrs, titles)):
    axes[idx].contour(XX, YY, np.log(ZZ + 1), levels=30, cmap='viridis', alpha=0.5)
    axes[idx].scatter(1, 1, c='red', marker='*', s=200, zorder=5, label='Minimum')
    
    path = gradient_descent_path(start, lr, 500)
    axes[idx].plot(path[:, 0], path[:, 1], 'r.-', markersize=2, lw=1, alpha=0.7)
    axes[idx].scatter(path[0, 0], path[0, 1], c='green', s=100, zorder=5, marker='o')
    axes[idx].set_title(title, fontsize=11)
    axes[idx].set_xlim(-2, 2)
    axes[idx].set_ylim(-1, 3)

plt.suptitle('Gradient Descent: Effect of Learning Rate', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Learning rate is crucial:')
print('  Too small → convergence is very slow')
print('  Too large → oscillation or divergence')
print('  Just right → efficient convergence to minimum')

---
## 5. Building Neural Networks with PyTorch

**PyTorch** is the most popular deep learning framework. It provides:
- **Tensors**: GPU-accelerated arrays (like NumPy on steroids)
- **Autograd**: automatic differentiation (no manual gradients!)
- **nn.Module**: composable building blocks for networks
- **Optimizers**: SGD, Adam, etc.

### 5.1 PyTorch Basics: Tensors and Autograd

Let's start with the fundamentals of PyTorch.

In [ ]:
if TORCH_AVAILABLE:
    # Tensors are like NumPy arrays
    x = torch.tensor([1.0, 2.0, 3.0])
    print(f'Tensor: {x}')
    print(f'Shape: {x.shape}, Dtype: {x.dtype}')
    
    # From NumPy
    np_array = np.array([[1, 2], [3, 4]], dtype=np.float32)
    t = torch.from_numpy(np_array)
    print(f'\nFrom NumPy: {t}')
    
    # Autograd: automatic differentiation!
    x = torch.tensor(2.0, requires_grad=True)
    y = x**2 + 3*x + 1  # y = x² + 3x + 1
    y.backward()          # compute dy/dx
    print(f'\nAutograd demo:')
    print(f'  y = x² + 3x + 1, at x=2')
    print(f'  y = {y.item():.1f}')
    print(f'  dy/dx = 2x + 3 = {x.grad.item():.1f} (computed automatically!)')
    
    # Matrix operations
    A = torch.randn(3, 4)
    B = torch.randn(4, 2)
    C = A @ B  # matrix multiply
    print(f'\nMatrix multiply: {A.shape} @ {B.shape} = {C.shape}')
else:
    print('PyTorch not available. Key concepts:')
    print('  torch.tensor() — create tensors (like np.array)')
    print('  requires_grad=True — enables automatic differentiation')
    print('  .backward() — computes all gradients automatically')

### 5.2 Building a Neural Network with nn.Module

PyTorch's `nn.Module` is the building block for all neural networks. Let's build a classifier for the Iris dataset.

In [ ]:
if TORCH_AVAILABLE:
    # Define the network architecture
    class IrisNet(nn.Module):
        def __init__(self):
            super().__init__()
            self.layers = nn.Sequential(
                nn.Linear(4, 16),    # input (4 features) → hidden (16 neurons)
                nn.ReLU(),           # activation
                nn.Linear(16, 8),    # hidden → hidden
                nn.ReLU(),
                nn.Linear(8, 3),     # hidden → output (3 classes)
            )
        
        def forward(self, x):
            return self.layers(x)
    
    # Prepare data
    iris = load_iris()
    X_train, X_test, y_train, y_test = train_test_split(
        iris.data, iris.target, test_size=0.2, random_state=42, stratify=iris.target
    )
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    
    # Convert to PyTorch tensors
    X_train_t = torch.FloatTensor(X_train_s)
    y_train_t = torch.LongTensor(y_train)
    X_test_t = torch.FloatTensor(X_test_s)
    y_test_t = torch.LongTensor(y_test)
    
    # Create model, loss function, optimizer
    model = IrisNet()
    criterion = nn.CrossEntropyLoss()  # standard loss for classification
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    
    print(f'Model architecture:\n{model}')
    n_params = sum(p.numel() for p in model.parameters())
    print(f'\nTotal parameters: {n_params}')
else:
    print('PyTorch model definition (nn.Module):')
    print('  class IrisNet(nn.Module):')
    print('    def __init__(self):')
    print('      self.layers = nn.Sequential(')
    print('        nn.Linear(4, 16), nn.ReLU(),')
    print('        nn.Linear(16, 8), nn.ReLU(),')
    print('        nn.Linear(8, 3))')

### 5.3 Training Loop

The PyTorch training loop follows a consistent pattern: forward → loss → backward → update.

In [ ]:
if TORCH_AVAILABLE:
    # Training loop
    torch.manual_seed(42)
    model = IrisNet()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    
    n_epochs = 200
    train_losses = []
    train_accs = []
    test_accs = []
    
    for epoch in range(n_epochs):
        # --- Training ---
        model.train()                          # set to training mode
        optimizer.zero_grad()                   # clear old gradients
        outputs = model(X_train_t)              # forward pass
        loss = criterion(outputs, y_train_t)    # compute loss
        loss.backward()                         # backward pass (compute gradients)
        optimizer.step()                        # update weights
        
        # --- Evaluation ---
        model.eval()                           # set to evaluation mode
        with torch.no_grad():                  # no gradients needed
            train_pred = outputs.argmax(dim=1)
            train_acc = (train_pred == y_train_t).float().mean()
            
            test_out = model(X_test_t)
            test_pred = test_out.argmax(dim=1)
            test_acc = (test_pred == y_test_t).float().mean()
        
        train_losses.append(loss.item())
        train_accs.append(train_acc.item())
        test_accs.append(test_acc.item())
        
        if (epoch + 1) % 50 == 0:
            print(f'Epoch {epoch+1:>3}/{n_epochs}: Loss={loss.item():.4f}, '
                  f'Train Acc={train_acc:.3f}, Test Acc={test_acc:.3f}')
    
    # Plot training history
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(train_losses, 'b-', lw=1)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training Loss')
    
    axes[1].plot(train_accs, 'b-', label='Train', lw=1)
    axes[1].plot(test_accs, 'r-', label='Test', lw=1)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Accuracy Over Training')
    axes[1].legend()
    
    plt.suptitle('Neural Network Training (Iris)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('Training loop pattern (PyTorch):')
    print('  for epoch in range(n_epochs):')
    print('      optimizer.zero_grad()      # clear gradients')
    print('      output = model(X_train)     # forward pass')
    print('      loss = criterion(output, y)  # compute loss')
    print('      loss.backward()             # compute gradients')
    print('      optimizer.step()            # update weights')

---
## 6. Convolutional Neural Networks (CNNs) — Overview

**CNNs** are designed for grid-like data (images, time series). They use **convolutional filters** that slide across the input to detect local patterns.

### Key Components

| Layer | Function | Output |
|-------|----------|--------|
| **Conv2d** | Applies learnable filters to detect features (edges, textures) | Feature maps |
| **ReLU** | Non-linear activation | Same size |
| **MaxPool2d** | Downsamples by taking max in each window | Smaller feature maps |
| **Flatten** | Reshapes 2D maps to 1D vector | Vector |
| **Linear** | Fully connected layer for classification | Class scores |

### 6.1 CNN Architecture Visualization

Let's define a simple CNN and visualize its structure.

In [ ]:
if TORCH_AVAILABLE:
    class SimpleCNN(nn.Module):
        """Simple CNN for 8x8 grayscale images (like sklearn digits)."""
        def __init__(self, n_classes=10):
            super().__init__()
            self.features = nn.Sequential(
                nn.Conv2d(1, 16, kernel_size=3, padding=1),  # 1→16 channels
                nn.ReLU(),
                nn.MaxPool2d(2),                              # 8x8 → 4x4
                nn.Conv2d(16, 32, kernel_size=3, padding=1), # 16→32 channels
                nn.ReLU(),
                nn.MaxPool2d(2),                              # 4x4 → 2x2
            )
            self.classifier = nn.Sequential(
                nn.Flatten(),
                nn.Linear(32 * 2 * 2, 64),
                nn.ReLU(),
                nn.Linear(64, n_classes)
            )
        
        def forward(self, x):
            x = self.features(x)
            x = self.classifier(x)
            return x
    
    cnn = SimpleCNN()
    print('CNN Architecture:')
    print(cnn)
    
    # Count parameters
    total = sum(p.numel() for p in cnn.parameters())
    trainable = sum(p.numel() for p in cnn.parameters() if p.requires_grad)
    print(f'\nTotal parameters: {total:,}')
    print(f'Trainable parameters: {trainable:,}')
    
    # Show data flow
    dummy = torch.randn(1, 1, 8, 8)  # batch=1, channels=1, 8x8
    print(f'\nData flow through layers:')
    x = dummy
    for name, layer in list(cnn.features.named_children()) + list(cnn.classifier.named_children()):
        x = layer(x)
        print(f'  After {layer.__class__.__name__}: {list(x.shape)}')
else:
    print('CNN layers explained:')
    print('  Conv2d(1, 16, 3): 1 input channel → 16 output channels, 3x3 filter')
    print('  MaxPool2d(2): reduce spatial size by half')
    print('  Flatten: reshape 2D feature maps to 1D vector')
    print('  Linear: standard fully connected layer')

---
## 7. Recurrent Neural Networks (RNNs) — Overview

**RNNs** are designed for sequential data (text, time series). They maintain a **hidden state** that carries information from previous time steps.

### RNN Variants

| Architecture | Key Feature | Use Case |
|-------------|-------------|----------|
| **Vanilla RNN** | Simple hidden state loop | Short sequences |
| **LSTM** | Gates to control information flow | Long sequences, text |
| **GRU** | Simplified LSTM (fewer gates) | Balanced efficiency |
| **Transformer** | Attention mechanism (no recurrence) | State-of-the-art for NLP |

### The Vanishing Gradient Problem

Vanilla RNNs struggle with long sequences because gradients shrink exponentially during backpropagation. **LSTMs** solve this with gating mechanisms.

### 7.1 LSTM for Sequence Prediction

Let's build a simple LSTM that learns a sine wave pattern.

In [ ]:
if TORCH_AVAILABLE:
    # Generate sine wave data
    t = np.linspace(0, 20*np.pi, 1000)
    data = np.sin(t)
    
    # Create sequences: use 20 past values to predict the next one
    seq_len = 20
    X_seq, y_seq = [], []
    for i in range(len(data) - seq_len):
        X_seq.append(data[i:i+seq_len])
        y_seq.append(data[i+seq_len])
    
    X_seq = torch.FloatTensor(np.array(X_seq)).unsqueeze(-1)  # (N, seq_len, 1)
    y_seq = torch.FloatTensor(np.array(y_seq)).unsqueeze(-1)  # (N, 1)
    
    # Split
    split = 800
    X_train_seq, X_test_seq = X_seq[:split], X_seq[split:]
    y_train_seq, y_test_seq = y_seq[:split], y_seq[split:]
    
    # Define LSTM model
    class LSTMPredictor(nn.Module):
        def __init__(self, hidden_size=32):
            super().__init__()
            self.lstm = nn.LSTM(input_size=1, hidden_size=hidden_size, batch_first=True)
            self.fc = nn.Linear(hidden_size, 1)
        
        def forward(self, x):
            lstm_out, _ = self.lstm(x)       # lstm_out: (batch, seq, hidden)
            last_hidden = lstm_out[:, -1, :]  # take last time step
            return self.fc(last_hidden)
    
    # Train
    torch.manual_seed(42)
    lstm_model = LSTMPredictor(hidden_size=32)
    optimizer = optim.Adam(lstm_model.parameters(), lr=0.005)
    criterion = nn.MSELoss()
    
    for epoch in range(100):
        lstm_model.train()
        optimizer.zero_grad()
        pred = lstm_model(X_train_seq)
        loss = criterion(pred, y_train_seq)
        loss.backward()
        optimizer.step()
    
    # Evaluate
    lstm_model.eval()
    with torch.no_grad():
        test_pred = lstm_model(X_test_seq).numpy()
    
    # Plot
    fig, ax = plt.subplots(figsize=(12, 4))
    test_range = range(split+seq_len, len(data))
    ax.plot(test_range, y_test_seq.numpy(), 'b-', lw=1.5, label='True')
    ax.plot(test_range, test_pred, 'r--', lw=1.5, label='LSTM Predicted')
    ax.set_xlabel('Time Step')
    ax.set_ylabel('Value')
    ax.set_title('LSTM: Sine Wave Prediction')
    ax.legend()
    plt.tight_layout()
    plt.show()
    
    mse = criterion(torch.FloatTensor(test_pred), y_test_seq).item()
    print(f'Test MSE: {mse:.6f}')
else:
    print('LSTM model structure:')
    print('  nn.LSTM(input_size=1, hidden_size=32)  # recurrent layer')
    print('  nn.Linear(32, 1)  # output layer')
    print('  LSTMs are powerful for time series and text sequences.')

---
## 8. The Transformer Architecture

The **Transformer** (2017) revolutionized deep learning by replacing recurrence with **self-attention**. It processes all positions in parallel, making it much faster and more effective.

### Key Innovation: Self-Attention

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Where Q (Query), K (Key), V (Value) are linear projections of the input.

**Intuition:** Each word "attends" to all other words, learning which ones are most relevant in context.

### Transformer Components

| Component | Function |
|-----------|----------|
| **Multi-Head Attention** | Multiple attention patterns in parallel |
| **Feed-Forward Network** | Process each position independently |
| **Layer Normalization** | Stabilize training |
| **Positional Encoding** | Inject sequence order information |
| **Residual Connections** | Help gradient flow in deep networks |

### 8.1 Simplified Self-Attention

Let's implement a simplified version of self-attention to understand the mechanism.

In [ ]:
def self_attention(X, d_k=None):
    """
    Simplified self-attention (single head, no learned projections).
    
    X: input matrix (seq_len, d_model)
    Returns: attended output, attention weights
    """
    if d_k is None:
        d_k = X.shape[1]
    
    # Q, K, V are all the same (self-attention)
    Q = K = V = X
    
    # Compute attention scores
    scores = Q @ K.T / np.sqrt(d_k)  # scale by sqrt(d_k)
    
    # Softmax to get attention weights (each row sums to 1)
    exp_scores = np.exp(scores - scores.max(axis=-1, keepdims=True))
    weights = exp_scores / exp_scores.sum(axis=-1, keepdims=True)
    
    # Weighted sum of values
    output = weights @ V
    
    return output, weights

# Demo: 4 words with 3-dimensional embeddings
words = ['The', 'cat', 'sat', 'down']
np.random.seed(42)
X = np.random.randn(4, 3)  # 4 words, 3-dim embeddings

output, attention = self_attention(X)

# Visualize attention weights
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.heatmap(attention, annot=True, fmt='.3f', cmap='Blues',
            xticklabels=words, yticklabels=words, ax=axes[0])
axes[0].set_title('Self-Attention Weights')
axes[0].set_xlabel('Key (attending to)')
axes[0].set_ylabel('Query (from)')

# Show how attention changes the representations
data_before_after = np.column_stack([X[:, 0], output[:, 0]])
df_ba = pd.DataFrame(data_before_after, index=words, columns=['Before', 'After'])
df_ba.plot(kind='bar', ax=axes[1], color=['steelblue', 'coral'])
axes[1].set_title('Embeddings Before/After Attention (dim 1)')
axes[1].set_xticklabels(words, rotation=0)

plt.suptitle('Self-Attention Mechanism', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Each word\'s new representation is a weighted combination of all words.')
print('Attention weights show which words are most relevant to each other.')

---
## 9. Large Language Models (LLMs)

**LLMs** are massive Transformer-based models trained to predict the next token in text. Through this simple objective, they learn grammar, facts, reasoning, and even code.

### How LLMs Work

1. **Tokenization**: Text → tokens (subwords): `"Hello world" → ["Hello", " world"]`
2. **Embedding**: Tokens → dense vectors
3. **Transformer layers**: Process through attention + feed-forward
4. **Output**: Probability distribution over next token
5. **Generation**: Sample or pick the most likely next token, repeat

### LLM Landscape (2025)

| Model | Organization | Parameters | Key Feature |
|-------|-------------|------------|-------------|
| GPT-4 | OpenAI | ~1.8T (est.) | Multimodal (text + images) |
| Claude 3/4 | Anthropic | Undisclosed | Safety-focused, long context |
| Gemini | Google | Undisclosed | Multimodal, search integrated |
| LLaMA 3 | Meta | 8B–405B | Open weights |
| Mistral | Mistral AI | 7B–8×22B | Efficient, open |

### Key Concepts

| Term | Meaning |
|------|---------|
| **Pre-training** | Learning from massive text corpora (unsupervised) |
| **Fine-tuning** | Adapting to specific tasks with labeled data |
| **RLHF** | Reinforcement Learning from Human Feedback (alignment) |
| **Prompt engineering** | Crafting inputs to get desired outputs |
| **Context window** | Maximum input length (e.g., 128K tokens) |
| **Temperature** | Controls randomness in generation (0=deterministic, 1=creative) |

### 9.1 Tokenization Demo

Let's see how text gets converted to tokens — the fundamental unit of LLM processing.

In [ ]:
# Simple word-level tokenizer (real LLMs use subword tokenizers like BPE)
class SimpleTokenizer:
    """A basic word-level tokenizer for demonstration."""
    
    def __init__(self):
        self.word2idx = {}
        self.idx2word = {}
        self.vocab_size = 0
    
    def fit(self, texts):
        """Build vocabulary from texts."""
        words = set()
        for text in texts:
            words.update(text.lower().split())
        
        self.word2idx = {w: i for i, w in enumerate(sorted(words))}
        self.idx2word = {i: w for w, i in self.word2idx.items()}
        self.vocab_size = len(self.word2idx)
    
    def encode(self, text):
        """Convert text to token IDs."""
        return [self.word2idx.get(w, -1) for w in text.lower().split()]
    
    def decode(self, ids):
        """Convert token IDs back to text."""
        return ' '.join(self.idx2word.get(i, '<UNK>') for i in ids)

# Demo
texts = [
    'Statistics is the science of learning from data',
    'Machine learning uses data to make predictions',
    'Deep learning is a subset of machine learning',
    'Neural networks learn patterns from data',
]

tokenizer = SimpleTokenizer()
tokenizer.fit(texts)

print(f'Vocabulary size: {tokenizer.vocab_size}')
print(f'\nVocabulary: {list(tokenizer.word2idx.keys())[:15]}...')

test_text = 'Deep learning uses neural networks'
tokens = tokenizer.encode(test_text)
decoded = tokenizer.decode(tokens)

print(f'\nInput: "{test_text}"')
print(f'Tokens: {tokens}')
print(f'Decoded: "{decoded}"')
print(f'\nNote: Real LLMs use BPE/SentencePiece with 32K-100K+ vocab size')
print('and handle subwords (e.g., "unbelievable" → ["un", "believ", "able"])')

### 9.2 Simulating Next-Token Prediction

Let's simulate how an LLM generates text by predicting one token at a time.

In [ ]:
def simulate_llm_generation(prompt, temperature=1.0, max_tokens=10):
    """
    Simulate LLM text generation (using a simple bigram model).
    This demonstrates the concept, not actual LLM quality.
    """
    # Build simple bigram probability table from our texts
    bigrams = {}
    for text in texts:
        words = text.lower().split()
        for i in range(len(words) - 1):
            if words[i] not in bigrams:
                bigrams[words[i]] = []
            bigrams[words[i]].append(words[i+1])
    
    # Generate
    generated = prompt.lower().split()
    print(f'Prompt: "{prompt}"')
    print(f'Temperature: {temperature}')
    print(f'Generation (step by step):')
    
    for step in range(max_tokens):
        current_word = generated[-1]
        if current_word not in bigrams:
            break
        
        # Get candidate next words and their probabilities
        candidates = bigrams[current_word]
        unique_words = list(set(candidates))
        counts = np.array([candidates.count(w) for w in unique_words], dtype=float)
        
        # Apply temperature
        logits = np.log(counts / counts.sum() + 1e-8)
        probs = np.exp(logits / temperature)
        probs = probs / probs.sum()
        
        # Sample
        next_word = np.random.choice(unique_words, p=probs)
        generated.append(next_word)
        
        prob_str = ', '.join(f'{w}({p:.2f})' for w, p in zip(unique_words, probs))
        print(f'  Step {step+1}: P(next|{current_word}) = [{prob_str}] → "{next_word}"')
    
    return ' '.join(generated)

np.random.seed(42)
result = simulate_llm_generation('machine', temperature=1.0, max_tokens=5)
print(f'\nFull output: "{result}"')

print('\n--- Temperature comparison ---')
for temp in [0.1, 0.5, 1.0, 2.0]:
    np.random.seed(42)
    words = ['machine']
    # Quick generation without step-by-step output
    bigrams_quick = {}
    for text in texts:
        ws = text.lower().split()
        for i in range(len(ws) - 1):
            bigrams_quick.setdefault(ws[i], []).append(ws[i+1])
    for _ in range(4):
        if words[-1] in bigrams_quick:
            cands = bigrams_quick[words[-1]]
            uq = list(set(cands))
            c = np.array([cands.count(w) for w in uq], dtype=float)
            logits = np.log(c/c.sum()+1e-8)
            p = np.exp(logits/temp); p = p/p.sum()
            words.append(np.random.choice(uq, p=p))
    print(f'  T={temp}: {" ".join(words)}')

---
## 10. Using LLMs via APIs

In practice, we use LLMs through APIs rather than training them ourselves. Here's how to interact with LLM APIs for data analysis tasks.

### 10.1 API Request Structure

Most LLM APIs follow a similar pattern.

In [ ]:
# Template for calling an LLM API (demonstration — requires API key)
import json

# This is the general structure for calling Claude, GPT-4, etc.
api_request_template = {
    'model': 'claude-sonnet-4-20250514',
    'max_tokens': 1024,
    'messages': [
        {
            'role': 'user',
            'content': 'Analyze this dataset and suggest statistical tests: '
                       'Two groups of students (n=30 each) with test scores. '
                       'Group A mean=75.3, std=8.2. Group B mean=82.1, std=7.5.'
        }
    ]
}

print('LLM API Request Structure:')
print(json.dumps(api_request_template, indent=2))

print('\n--- How to call the API (pseudocode) ---')
print('import anthropic  # or openai')
print('client = anthropic.Anthropic(api_key="your-key")')
print('response = client.messages.create(**api_request_template)')
print('print(response.content[0].text)')

print('\n--- Common LLM use cases for statistics ---')
use_cases = [
    'Explain statistical results in plain language',
    'Suggest appropriate statistical tests',
    'Generate Python code for analysis',
    'Interpret p-values and effect sizes',
    'Review methodology for errors',
    'Summarize research papers',
]
for i, case in enumerate(use_cases, 1):
    print(f'  {i}. {case}')

### 10.2 Prompt Engineering for Data Analysis

Effective prompts help LLMs give better statistical advice. Let's look at prompt patterns.

In [ ]:
# Prompt engineering patterns for statistical analysis
prompts = {
    'Basic Question': 
        'What test should I use to compare two groups?',
    
    'Structured Prompt (Better)': 
        '''I have the following data:
- Two independent groups (treatment vs control)
- Sample sizes: n1=45, n2=50  
- Outcome: continuous (reaction time in ms)
- Data appears normally distributed (Shapiro p=0.12, 0.08)
- Equal variances (Levene p=0.34)

Which statistical test should I use? Please explain:
1. The recommended test and why
2. Assumptions I should verify
3. Python code using scipy.stats
4. How to interpret the results''',
    
    'Code Generation Prompt':
        '''Write Python code to:
1. Load a CSV with columns: age, income, education, purchased
2. Split into train/test (80/20)
3. Train a Random Forest classifier
4. Print classification report and feature importance
5. Plot the confusion matrix

Use scikit-learn. Add comments explaining each step.'''
}

print('=== Prompt Engineering for Statistics ===')
for name, prompt in prompts.items():
    print(f'\n--- {name} ---')
    print(prompt[:200] + ('...' if len(prompt) > 200 else ''))

print('\n=== Tips for Better Prompts ===')
tips = [
    'Provide context: sample sizes, data types, distributions',
    'Be specific: ask for code, interpretation, or both',
    'Include constraints: "use scipy.stats", "assume alpha=0.05"',
    'Ask for explanations: "explain why this test is appropriate"',
    'Verify outputs: always check LLM-generated code by running it',
]
for i, tip in enumerate(tips, 1):
    print(f'  {i}. {tip}')

---
## 11. Practical Example: Image Classification with PyTorch

Let's build a complete neural network pipeline to classify handwritten digits from the sklearn digits dataset.

### Step 1: Load and Explore the Data

In [ ]:
# Load handwritten digits
digits = load_digits()
X_digits = digits.data  # 8x8 images flattened to 64 features
y_digits = digits.target

print(f'Dataset: {X_digits.shape[0]} images, {X_digits.shape[1]} features (8x8 pixels)')
print(f'Classes: {np.unique(y_digits)} (digits 0-9)')

# Visualize sample digits
fig, axes = plt.subplots(2, 10, figsize=(14, 3))
for digit in range(10):
    idx = np.where(y_digits == digit)[0][0]  # first occurrence
    axes[0][digit].imshow(digits.images[idx], cmap='gray_r')
    axes[0][digit].set_title(str(digit), fontsize=11)
    axes[0][digit].axis('off')
    
    idx2 = np.where(y_digits == digit)[0][5]  # another example
    axes[1][digit].imshow(digits.images[idx2], cmap='gray_r')
    axes[1][digit].axis('off')

plt.suptitle('Handwritten Digits (8x8 pixels)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Step 2: Prepare Data and Build Model

We preprocess the data and define a neural network for digit classification.

In [ ]:
# Prepare data
X_train, X_test, y_train, y_test = train_test_split(
    X_digits, y_digits, test_size=0.2, random_state=42, stratify=y_digits
)

# Scale to [0, 1]
X_train_s = X_train / 16.0  # pixel values are 0-16
X_test_s = X_test / 16.0

if TORCH_AVAILABLE:
    # Convert to tensors
    X_train_t = torch.FloatTensor(X_train_s)
    y_train_t = torch.LongTensor(y_train)
    X_test_t = torch.FloatTensor(X_test_s)
    y_test_t = torch.LongTensor(y_test)
    
    # Create DataLoader for mini-batch training
    train_dataset = TensorDataset(X_train_t, y_train_t)
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    
    # Define model
    class DigitNet(nn.Module):
        def __init__(self):
            super().__init__()
            self.network = nn.Sequential(
                nn.Linear(64, 128),
                nn.ReLU(),
                nn.Dropout(0.2),      # regularization
                nn.Linear(128, 64),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(64, 10),     # 10 digit classes
            )
        
        def forward(self, x):
            return self.network(x)
    
    torch.manual_seed(42)
    model = DigitNet()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    print(f'Model: {sum(p.numel() for p in model.parameters()):,} parameters')
    print(model)
else:
    print('Model architecture: 64 → 128 → 64 → 10')
    print('With ReLU activations and Dropout regularization')

### Step 3: Train with Mini-Batch Gradient Descent

We train using mini-batches and track both training and test performance.

In [ ]:
if TORCH_AVAILABLE:
    n_epochs = 50
    history = {'train_loss': [], 'train_acc': [], 'test_acc': []}
    
    for epoch in range(n_epochs):
        # --- Training ---
        model.train()
        epoch_loss = 0
        correct = 0
        total = 0
        
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item() * len(y_batch)
            correct += (outputs.argmax(1) == y_batch).sum().item()
            total += len(y_batch)
        
        train_loss = epoch_loss / total
        train_acc = correct / total
        
        # --- Evaluation ---
        model.eval()
        with torch.no_grad():
            test_out = model(X_test_t)
            test_acc = (test_out.argmax(1) == y_test_t).float().mean().item()
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)
        
        if (epoch + 1) % 10 == 0:
            print(f'Epoch {epoch+1:>3}/{n_epochs}: Loss={train_loss:.4f}, '
                  f'Train={train_acc:.3f}, Test={test_acc:.3f}')
    
    # Plot training history
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history['train_loss'], 'b-')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training Loss')
    
    axes[1].plot(history['train_acc'], 'b-', label='Train')
    axes[1].plot(history['test_acc'], 'r-', label='Test')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Accuracy')
    axes[1].legend()
    
    plt.suptitle('Digit Classification Training', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    # Use sklearn MLP as fallback
    from sklearn.neural_network import MLPClassifier
    model = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=200, random_state=42)
    model.fit(X_train_s, y_train)
    print(f'sklearn MLP test accuracy: {model.score(X_test_s, y_test):.3f}')

### Step 4: Evaluate and Visualize Results

Let's create a comprehensive evaluation dashboard.

In [ ]:
if TORCH_AVAILABLE:
    model.eval()
    with torch.no_grad():
        test_out = model(X_test_t)
        y_pred = test_out.argmax(1).numpy()
        y_proba = torch.softmax(test_out, dim=1).numpy()
else:
    y_pred = model.predict(X_test_s)
    y_proba = model.predict_proba(X_test_s)

# Classification report
print('Classification Report:')
print(classification_report(y_test, y_pred))

# Dashboard
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0,0],
            xticklabels=range(10), yticklabels=range(10))
axes[0,0].set_xlabel('Predicted')
axes[0,0].set_ylabel('Actual')
axes[0,0].set_title('Confusion Matrix')

# 2. Per-class accuracy
class_acc = cm.diagonal() / cm.sum(axis=1)
axes[0,1].bar(range(10), class_acc, color='steelblue')
axes[0,1].set_xlabel('Digit')
axes[0,1].set_ylabel('Accuracy')
axes[0,1].set_title('Per-Digit Accuracy')
axes[0,1].set_ylim(0.8, 1.05)

# 3. Sample predictions (correct and incorrect)
correct_mask = y_pred == y_test
wrong_mask = ~correct_mask
wrong_idx = np.where(wrong_mask)[0]

# Show some wrong predictions
n_show = min(8, len(wrong_idx))
for i in range(n_show):
    idx = wrong_idx[i]
    ax_sub = axes[1,0] if i < 4 else axes[1,1]
    # We'll use a different approach — just plot in one axes

axes[1,0].clear()
axes[1,1].clear()

# Show 5 correct and 5 incorrect predictions
correct_idx = np.where(correct_mask)[0][:5]
wrong_idx_show = wrong_idx[:5] if len(wrong_idx) >= 5 else wrong_idx

for i, idx in enumerate(correct_idx):
    ax_inset = axes[1,0].inset_axes([i*0.2, 0.1, 0.18, 0.8])
    ax_inset.imshow(X_test[idx].reshape(8, 8), cmap='gray_r')
    ax_inset.set_title(f'{y_pred[idx]}', fontsize=10, color='green')
    ax_inset.axis('off')
axes[1,0].set_title('Correct Predictions', fontsize=12, color='green')
axes[1,0].axis('off')

for i, idx in enumerate(wrong_idx_show):
    ax_inset = axes[1,1].inset_axes([i*0.2, 0.1, 0.18, 0.8])
    ax_inset.imshow(X_test[idx].reshape(8, 8), cmap='gray_r')
    ax_inset.set_title(f'{y_pred[idx]}({y_test[idx]})', fontsize=9, color='red')
    ax_inset.axis('off')
axes[1,1].set_title('Wrong Predictions (pred(true))', fontsize=12, color='red')
axes[1,1].axis('off')

plt.suptitle(f'Digit Classification Results — Accuracy: {accuracy_score(y_test, y_pred):.1%}',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 12. Summary

| Section | Key Concepts | Python Tools |
|---------|-------------|---------------|
| 1. ML to DL | Feature learning, GPU training, DL milestones | Conceptual |
| 2. Perceptron | Weights, bias, activation functions | NumPy implementation |
| 3. NN from Scratch | Forward pass, multi-layer, XOR problem | `SimpleNeuralNetwork` class |
| 4. Backpropagation | Chain rule, gradient descent, optimizers | NumPy, visualization |
| 5. PyTorch Basics | Tensors, autograd, nn.Module, training loop | `torch`, `nn.Module`, `optim.Adam` |
| 6. CNNs | Convolution, pooling, feature maps | `nn.Conv2d`, `nn.MaxPool2d` |
| 7. RNNs | Sequential data, LSTM, hidden state | `nn.LSTM` |
| 8. Transformers | Self-attention, Q/K/V, positional encoding | NumPy attention demo |
| 9. LLMs | Tokenization, pre-training, RLHF, temperature | Conceptual + demo |
| 10. LLM APIs | API structure, prompt engineering patterns | `anthropic` / `openai` |
| 11. Digit Classification | End-to-end PyTorch pipeline | Full training + evaluation |

## Homework

### Task 1: Neural Network from Scratch
Extend the `SimpleNeuralNetwork` class:
1. Add support for 2+ hidden layers.
2. Train on the make_moons dataset (300 points, noise=0.2).
3. Compare networks with 1, 2, and 3 hidden layers (8 neurons each).
4. Plot the decision boundary for each and report accuracy.

### Task 2: PyTorch Classification
Using the Wine dataset:
1. Build a PyTorch neural network (3+ layers).
2. Train for 200 epochs with Adam optimizer.
3. Plot training loss and accuracy curves.
4. Compare with a scikit-learn Random Forest — which is better?

### Task 3: Attention Visualization
Expand the self-attention demo:
1. Create a sentence of 6+ words with meaningful relationships.
2. Use random embeddings (dimension 8+) and compute self-attention.
3. Visualize the attention matrix as a heatmap.
4. Which word pairs have the highest attention? Does it make intuitive sense?

### Task 4: LLM Prompt Engineering
Write 3 detailed prompts for using an LLM to help with statistical analysis:
1. A prompt asking for test selection advice (include data description).
2. A prompt asking for Python code to perform a specific analysis.
3. A prompt asking for interpretation of results (provide numbers).
For each, explain what makes it a good prompt and what a bad version would look like.

---
## Course Wrap-Up

Congratulations! You've completed all the content weeks of **Applied Statistics with Python (2026)**. Over 16 weeks, we covered:

| Phase | Weeks | Topics |
|-------|-------|--------|
| **Foundations** | 1–4 | Python, NumPy, Pandas, Visualization |
| **Descriptive & Probability** | 5–6 | EDA, Probability, Distributions |
| **Inference** | 7–10 | Sampling, Hypothesis Testing, Regression, ANOVA |
| **Advanced Statistics** | 11–12 | Time Series, Bayesian Statistics |
| **Data Collection** | 13 | Web Scraping |
| **Machine Learning** | 14–16 | Supervised, Unsupervised, Deep Learning & LLMs |

**Weeks 17–18** will be dedicated to **Student Paper Replication Presentations**. Good luck with your projects!

---
*Applied Statistics with Python (2026) | Week 16 | Deep Learning & LLMs*